# 🦊 Kitsune NIDS: Quickstart Anomaly Detection Demo (Mirai Botnet)
**Academic Reference:** Y. Mirsky et al., *Kitsune: An Ensemble of Autoencoders for Online Network Intrusion Detection*, **NDSS 2018** (DOI: [10.14722/ndss.2018.23204](https://doi.org/10.14722/ndss.2018.23204)).  
**Organization:** [kitsune-research](https://github.com/kitsune-research/Kitsune-py) | **Release:** `v0.3.1`

---

### Overview & Mechanics
This notebook provides a fully reproducible demonstration of online, unsupervised network anomaly detection. 
The engine streams raw network packets, extracts 100 continuous statistical features in $\mathcal{O}(1)$ complexity across 5 damped temporal windows (AfterImage), dynamically clusters subspaces (corClust, $m \le 10$), and isolates anomalies via an ensemble of autoencoders (KitNET, $\mathcal{O}(k^2)$).

## 1. Environment Bootstrap & Data Pipeline
Installs dependencies in headless mode and provisions the Mirai botnet evaluation trace.

In [ ]:
import os
import zipfile
from pathlib import Path

# Check if running under Google Colab
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    print("[*] Google Colab environment detected. Initializing workspace...")
    if not Path("repo").exists():
        !git clone --depth 1 https://github.com/kitsune-research/Kitsune-py.git repo
    %cd repo
    !pip install --upgrade pip --quiet
    !pip install -e . --quiet

# Automatic dataset provisioning
DATA_FILE = Path("mirai.pcap")
if not DATA_FILE.exists():
    if Path("mirai.zip").exists():
        print("[*] Extracting local mirai.zip archive...")
        with zipfile.ZipFile("mirai.zip", 'r') as zip_ref:
            zip_ref.extractall(".")
    else:
        print("[*] Downloading compressed Mirai trace from upstream repository...")
        !wget -q -O mirai.zip https://github.com/kitsune-research/Kitsune-py/raw/main/mirai.zip
        with zipfile.ZipFile("mirai.zip", 'r') as zip_ref:
            zip_ref.extractall(".")

print(f"[+] Verification: Dataset mounted at {DATA_FILE.resolve()} | Exists: {DATA_FILE.exists()} | Size: {DATA_FILE.stat().st_size / (1024*1024):.2f} MB")

## 2. Streaming Ingestion & Real-Time Anomaly Scoring
Ingestion parameters:
* **`FM_grace_period = 2,000`:** Topological clustering window via corClust ($m \le 10$).
* **`AD_grace_period = 5,000`:** Manifold calibration via online single-pass SGD (`max_iter = 1`).
* **Evaluation Cap:** Processing the entire Mirai trace (~764,137 packets).

In [ ]:
import time
import numpy as np
from Kitsune import Kitsune

PACKET_LIMIT = np.inf  # Process entire PCAP (~764k packets)
MAX_AE = 10
FM_GRACE = 2000
AD_GRACE = 5000

print(f"[*] Instantiating Kitsune NIDS Engine...")
k_engine = Kitsune(
    file_path=str(DATA_FILE),
    limit=PACKET_LIMIT,
    max_autoencoder_size=MAX_AE,
    FM_grace_period=FM_GRACE,
    AD_grace_period=AD_GRACE
)

rmse_scores = []
start_time = time.perf_counter()

print("[*] Processing streaming packet flow...")
count = 0
while True:
    rmse = k_engine.proc_next_packet()
    if rmse == -1:
        break
    rmse_scores.append(rmse)
    count += 1
    if count % 25000 == 0:
        elapsed = time.perf_counter() - start_time
        fps = count / elapsed
        print(f"    -> Ingested {count:7d} packets | Current RMSE: {rmse:8.4f} | Throughput: {fps:6.1f} pkt/sec")

total_time = time.perf_counter() - start_time
print(f"\n[+] Execution complete: {len(rmse_scores)} packets scored in {total_time:.2f}s ({len(rmse_scores)/total_time:.1f} pkt/s)")

## 3. Dynamic Thresholding (Analytical Log-Normal Fit)
Models the reconstruction residue over the stable baseline ($10,000 \le t \le 50,000$) using a log-normal distribution:
$$\ln(\mathrm{RMSE}) \sim \mathcal{N}(\mu_{\ln}, \sigma_{\ln}^2) \implies \tau = \exp(\mu_{\ln} + k \cdot \sigma_{\ln})$$
With $k = 3.0902$, the theoretical False Positive Rate is bounded to $\mathrm{FPR} \le 0.001$.

In [ ]:
# Baseline extraído del tramo benigno en exec-mode limpio (paquetes 10,000 a 50,000)
baseline_window = [s for s in rmse_scores[10000:50000] if s > 0]

log_baseline = np.log(baseline_window)
mu_log = np.mean(log_baseline)
sigma_log = np.std(log_baseline)
k_risk = 3.0902
tau = np.exp(mu_log + k_risk * sigma_log)

print(f"[+] Calibrated Manifold Baseline:")
print(f"    - Mean [ln(RMSE)]     : {mu_log:.4f}")
print(f"    - StdDev [ln(RMSE)]   : {sigma_log:.4f}")
print(f"    - Decision Cutoff (tau): {tau:.4f}")

## 4. Anomaly Visualization & Attack Trajectory
Displays the continuous reconstruction error trajectory against the calibrated cutoff $\tau$.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5.5), dpi=140)
plt.style.use('dark_background')

x_axis = np.arange(len(rmse_scores))
plt.plot(x_axis, rmse_scores, color='#38bdf8', lw=1.2, label='KitNET RMSE Residue', alpha=0.9)

# Umbral log-normal tau con macro LaTeX canónico \leq para evitar ParseFatalException
plt.axhline(y=tau, color='#f43f5e', linestyle='--', lw=1.8, label=rf'Decision Cutoff $\tau = {tau:.4f}$ (FPR $\leq 10^{{-3}}$)')

plt.axvspan(0, FM_GRACE, color='#64748b', alpha=0.25, label='corClust Feature Mapping')
plt.axvspan(FM_GRACE, AD_GRACE, color='#eab308', alpha=0.15, label='Online SGD Calibration')

# El ataque Mirai real arranca masivamente tras la estabilización benigna (~100,000)
ATTACK_START = 100000
if len(rmse_scores) > ATTACK_START:
    plt.axvspan(ATTACK_START, len(rmse_scores), color='#ef4444', alpha=0.20, label='Mirai Attack Phase')

plt.yscale('log')
plt.title('Kitsune NIDS: Online Streaming Anomaly Detection (Mirai Botnet Trace)', fontsize=13, pad=12)
plt.xlabel('Ingested Packets (Continuous Stream)', fontsize=10)
plt.ylabel('RMSE Reconstruction Error (Log Scale)', fontsize=10)
plt.grid(True, which='both', ls=':', alpha=0.25)
plt.legend(loc='upper left', framealpha=0.85)
plt.tight_layout()
plt.show()

## 5. Forensic Metrics & Signal-to-Noise Ratio (SNR)
Evaluates separation against the benign noise floor.

In [ ]:
ATTACK_START = 100000
attack_scores = np.array(rmse_scores[ATTACK_START:]) if len(rmse_scores) > ATTACK_START else np.array([])
baseline_scores = np.array(rmse_scores[10000:50000])

if len(attack_scores) > 0:
    detected = np.sum(attack_scores > tau)
    ratio = (detected / len(attack_scores)) * 100
    snr = np.median(attack_scores) / np.median(baseline_scores)
    peak_snr = np.max(attack_scores) / np.median(baseline_scores)

    print("=" * 62)
    print(" KITSUNE NIDS: FORENSIC SEPARATION SUMMARY")
    print("=" * 62)
    print(f" Attack packets evaluated        : {len(attack_scores):,}")
    print(f" Anomaly detections (> tau)       : {detected:,} ({ratio:.2f}%)")
    print(f" Median Separation vs Baseline    : {snr:.2f}x")
    print(f" Peak Anomaly vs Noise Floor       : {peak_snr:.2f}x")
    print("=" * 62)
else:
    print("[*] Not enough packets to compute attack evaluation phase.")